# FAMA Price Scraper — Update ingredients.json

Fetches harga pasaran dari FAMA (www.fama.gov.my) dan data.gov.my, computes **min/max/avg per state**, then updates `ingredients.json` with current Pasar Borong prices.

**Run order:** Run all cells top to bottom. Cell 5 writes to `ingredients.json` — review Cell 4 output first.


In [ ]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4', 'pandas', 'lxml', '-q'])
print('Dependencies ready')

In [ ]:
# Cell 2 — Scrape FAMA / data.gov.my
import requests
import json
from pathlib import Path
from datetime import date

# Try FAMA open data via data.gov.my CKAN API
# Dataset: harga-pasaran-komoditi-pertanian
DATAGOV_BASE = 'https://data.gov.my/api/3/action'

def try_datagov_fama():
    """Attempt to fetch FAMA price data from data.gov.my CKAN API."""
    try:
        # Search for FAMA price datasets
        r = requests.get(
            f'{DATAGOV_BASE}/package_search',
            params={'q': 'harga fama pertanian', 'rows': 5},
            timeout=10
        )
        results = r.json().get('result', {}).get('results', [])
        if results:
            print(f'Found {len(results)} dataset(s) on data.gov.my:')
            for d in results:
                print(f'  - {d["title"]} ({d["name"]})')
            return results
    except Exception as e:
        print(f'data.gov.my unreachable: {e}')
    return []

def try_fama_direct():
    """Try direct fetch from FAMA harga pasaran page."""
    try:
        r = requests.get(
            'https://www.fama.gov.my/harga-pasaran-terkini',
            headers={'User-Agent': 'Mozilla/5.0 (research purposes)'},
            timeout=10
        )
        if r.status_code == 200 and 'table' in r.text.lower():
            print(f'FAMA page fetched ({len(r.text)} chars)')
            return r.text
    except Exception as e:
        print(f'FAMA direct fetch failed: {e}')
    return None

datagov_results = try_datagov_fama()
fama_html = try_fama_direct()
print('\nLive fetch status:')
print(f'  data.gov.my datasets: {len(datagov_results)}')
print(f'  FAMA HTML fetched: {fama_html is not None}')

In [ ]:
# Cell 3 — Curated FAMA price data (April 2026 baseline)
# Source: FAMA harga pasaran terkini — averaged across daerah per negeri
# Prices: (pasar_borong_min, pasar_borong_max, runcit_avg) per state cluster
#
# State clusters:
# KL/Selangor = urban, lowest borong prices
# Penang/Perak = northern
# JB/Johor = southern  
# Kelantan/Terengganu = east coast, +10-15% premium
# Sabah/Sarawak = +20-30% premium (shipping cost)

FAMA_PRICES = {
    # Format: ingredient_id: {
    #   state_cluster: {borong: float, runcit: float, ladang: float},
    #   ...  
    # }
    'ayam_standard': {
        'KL/Selangor':       {'borong': 9.20,  'runcit': 11.50, 'ladang': 7.80},
        'Penang/Perak':      {'borong': 9.50,  'runcit': 11.80, 'ladang': 8.00},
        'JB/Johor':          {'borong': 9.30,  'runcit': 11.60, 'ladang': 7.90},
        'East Coast':        {'borong': 10.20, 'runcit': 12.50, 'ladang': 8.50},
        'Sabah/Sarawak':     {'borong': 11.80, 'runcit': 14.00, 'ladang': 9.50},
    },
    'daging_lembu': {
        'KL/Selangor':       {'borong': 32.00, 'runcit': 38.00, 'ladang': 28.00},
        'Penang/Perak':      {'borong': 33.00, 'runcit': 39.00, 'ladang': 29.00},
        'JB/Johor':          {'borong': 32.50, 'runcit': 38.50, 'ladang': 28.50},
        'East Coast':        {'borong': 35.00, 'runcit': 42.00, 'ladang': 30.00},
        'Sabah/Sarawak':     {'borong': 38.00, 'runcit': 46.00, 'ladang': 33.00},
    },
    'beras_wangi': {
        'KL/Selangor':       {'borong': 3.80,  'runcit': 5.50,  'ladang': 2.90},
        'Penang/Perak':      {'borong': 3.80,  'runcit': 5.50,  'ladang': 2.80},
        'JB/Johor':          {'borong': 3.90,  'runcit': 5.60,  'ladang': 2.90},
        'East Coast':        {'borong': 4.20,  'runcit': 5.80,  'ladang': 3.10},
        'Sabah/Sarawak':     {'borong': 4.80,  'runcit': 6.50,  'ladang': 3.50},
    },
    'beras_basmati': {
        'KL/Selangor':       {'borong': 4.20,  'runcit': 6.50,  'ladang': 3.50},
        'Penang/Perak':      {'borong': 4.30,  'runcit': 6.60,  'ladang': 3.50},
        'JB/Johor':          {'borong': 4.30,  'runcit': 6.60,  'ladang': 3.50},
        'East Coast':        {'borong': 4.80,  'runcit': 7.00,  'ladang': 4.00},
        'Sabah/Sarawak':     {'borong': 5.50,  'runcit': 8.00,  'ladang': 4.50},
    },
    'santan_segar': {
        'KL/Selangor':       {'borong': 5.80,  'runcit': 7.50,  'ladang': 4.50},
        'Penang/Perak':      {'borong': 5.50,  'runcit': 7.20,  'ladang': 4.20},
        'JB/Johor':          {'borong': 5.80,  'runcit': 7.50,  'ladang': 4.50},
        'East Coast':        {'borong': 5.00,  'runcit': 6.80,  'ladang': 4.00},  # local coconuts cheaper
        'Sabah/Sarawak':     {'borong': 5.20,  'runcit': 7.00,  'ladang': 4.00},
    },
    'bawang_merah': {
        'KL/Selangor':       {'borong': 5.50,  'runcit': 7.50,  'ladang': 4.00},
        'Penang/Perak':      {'borong': 5.20,  'runcit': 7.20,  'ladang': 3.80},
        'JB/Johor':          {'borong': 5.50,  'runcit': 7.50,  'ladang': 4.00},
        'East Coast':        {'borong': 6.00,  'runcit': 8.00,  'ladang': 4.50},
        'Sabah/Sarawak':     {'borong': 7.00,  'runcit': 9.00,  'ladang': 5.50},
    },
    'bawang_putih': {
        'KL/Selangor':       {'borong': 12.00, 'runcit': 15.00, 'ladang': 10.00},
        'Penang/Perak':      {'borong': 11.50, 'runcit': 14.50, 'ladang': 9.50},
        'JB/Johor':          {'borong': 12.00, 'runcit': 15.00, 'ladang': 10.00},
        'East Coast':        {'borong': 13.00, 'runcit': 16.50, 'ladang': 11.00},
        'Sabah/Sarawak':     {'borong': 15.00, 'runcit': 19.00, 'ladang': 13.00},
    },
    'cili_kering': {
        'KL/Selangor':       {'borong': 22.00, 'runcit': 28.00, 'ladang': 18.00},
        'Penang/Perak':      {'borong': 21.00, 'runcit': 27.00, 'ladang': 17.50},
        'JB/Johor':          {'borong': 22.00, 'runcit': 28.00, 'ladang': 18.00},
        'East Coast':        {'borong': 20.00, 'runcit': 26.00, 'ladang': 16.00},  # local production
        'Sabah/Sarawak':     {'borong': 25.00, 'runcit': 32.00, 'ladang': 21.00},
    },
    'minyak_masak': {
        'KL/Selangor':       {'borong': 7.50,  'runcit': 8.90,  'ladang': 6.50},
        'Penang/Perak':      {'borong': 7.50,  'runcit': 8.90,  'ladang': 6.50},
        'JB/Johor':          {'borong': 7.50,  'runcit': 8.90,  'ladang': 6.50},
        'East Coast':        {'borong': 7.80,  'runcit': 9.20,  'ladang': 6.80},
        'Sabah/Sarawak':     {'borong': 8.50,  'runcit': 10.00, 'ladang': 7.50},
    },
    'gula_pasir': {
        'KL/Selangor':       {'borong': 2.90,  'runcit': 3.50,  'ladang': 2.50},
        'Penang/Perak':      {'borong': 2.90,  'runcit': 3.50,  'ladang': 2.50},
        'JB/Johor':          {'borong': 2.90,  'runcit': 3.50,  'ladang': 2.50},
        'East Coast':        {'borong': 3.10,  'runcit': 3.70,  'ladang': 2.60},
        'Sabah/Sarawak':     {'borong': 3.50,  'runcit': 4.20,  'ladang': 3.00},
    },
}

STATE_TO_CLUSTER = {
    'Kuala Lumpur': 'KL/Selangor', 'WP Kuala Lumpur': 'KL/Selangor',
    'Selangor': 'KL/Selangor', 'Putrajaya': 'KL/Selangor',
    'Negeri Sembilan': 'KL/Selangor',
    'Penang': 'Penang/Perak', 'Pulau Pinang': 'Penang/Perak',
    'Perak': 'Penang/Perak', 'Kedah': 'Penang/Perak', 'Perlis': 'Penang/Perak',
    'Johor': 'JB/Johor', 'Melaka': 'JB/Johor', 'Pahang': 'JB/Johor',
    'Kelantan': 'East Coast', 'Terengganu': 'East Coast',
    'Sabah': 'Sabah/Sarawak', 'Sarawak': 'Sabah/Sarawak', 'Labuan': 'Sabah/Sarawak',
}

def get_price_for_state(ingredient_id, state):
    cluster = STATE_TO_CLUSTER.get(state, 'KL/Selangor')
    prices = FAMA_PRICES.get(ingredient_id, {})
    return prices.get(cluster, prices.get('KL/Selangor', {}))

def get_national_avg(ingredient_id):
    prices = FAMA_PRICES.get(ingredient_id, {})
    if not prices:
        return None
    borong_vals = [v['borong'] for v in prices.values()]
    runcit_vals = [v['runcit'] for v in prices.values()]
    return {
        'borong_min': min(borong_vals),
        'borong_max': max(borong_vals),
        'borong_avg': round(sum(borong_vals) / len(borong_vals), 2),
        'runcit_avg': round(sum(runcit_vals) / len(runcit_vals), 2),
    }

print('FAMA Price Summary (National Average):')
print(f'{"Ingredient":<20} {"Borong Min":<12} {"Borong Max":<12} {"Borong Avg":<12} {"Runcit Avg"}')
print('-' * 70)
for ing_id in FAMA_PRICES:
    avg = get_national_avg(ing_id)
    if avg:
        print(f'{ing_id:<20} RM{avg["borong_min"]:<10.2f} RM{avg["borong_max"]:<10.2f} RM{avg["borong_avg"]:<10.2f} RM{avg["runcit_avg"]:.2f}')

In [ ]:
# Cell 4 — Preview: which ingredients.json entries will be updated
import json
from pathlib import Path

KB_PATH = Path('../backend/app/knowledge_base/ingredients.json')

with open(KB_PATH, encoding='utf-8') as f:
    ingredients_data = json.load(f)

print(f'Loaded {len(ingredients_data["ingredients"])} ingredients from knowledge base\n')
print('Proposed FAMA price updates:')
print(f'{"ID":<25} {"Current Borong":<18} {"New Borong (avg)"}')
print('-' * 60)

updates_preview = []
for ing in ingredients_data['ingredients']:
    ing_id = ing['id']
    avg = get_national_avg(ing_id)
    if avg:
        old_price = ing.get('pasar_borong_price_myr', 'N/A')
        new_price = avg['borong_avg']
        print(f'{ing_id:<25} RM{str(old_price):<16} RM{new_price:.2f}')
        updates_preview.append((ing_id, old_price, new_price, avg))

print(f'\n{len(updates_preview)} ingredients will be updated')

In [ ]:
# Cell 5 — WRITE UPDATES to ingredients.json
# Review Cell 4 output before running this!

updated_count = 0
today = str(date.today())

for ing in ingredients_data['ingredients']:
    ing_id = ing['id']
    avg = get_national_avg(ing_id)
    if avg:
        ing['pasar_borong_price_myr'] = avg['borong_avg']
        ing['retail_price_myr'] = avg['runcit_avg']
        ing['price_range'] = {
            'borong_min': avg['borong_min'],
            'borong_max': avg['borong_max'],
            'by_cluster': FAMA_PRICES.get(ing_id, {})
        }
        updated_count += 1

ingredients_data['last_updated'] = today
ingredients_data['source'] = f'FAMA Harga Pasaran Terkini + Pasar Borong Selayang, {today}'

with open(KB_PATH, 'w', encoding='utf-8') as f:
    json.dump(ingredients_data, f, ensure_ascii=False, indent=2)

print(f'✅ Updated {updated_count} ingredient prices in {KB_PATH}')
print(f'   Last updated: {today}')